In [ ]:
%pip install roboflow -q

## Upload dataset

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
V2_ROBOFLOW_API_KEY = userdata.get("V2_ROBOFLOW_API_KEY")

rf = Roboflow(api_key=V2_ROBOFLOW_API_KEY)
project = rf.workspace("caretech-v2").project("v2-caretech-combined-dataset")
dataset = project.version(1).download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to V2-CareTech-Combined-Dataset-1 in yolo26:: 100%|██████████| 58568/58568 [00:07<00:00, 8223.51it/s] 


## Sanity check dataset

In [ ]:
import os

DATASET_PATH = dataset.location

TRAIN_IMAGES = os.path.join(DATASET_PATH, "train/images")
TRAIN_LABELS = os.path.join(DATASET_PATH, "train/labels")

VAL_IMAGES = os.path.join(DATASET_PATH, "valid/images")
VAL_LABELS = os.path.join(DATASET_PATH, "valid/labels")

TEST_IMAGES = os.path.join(DATASET_PATH, "test/images")
TEST_LABELS = os.path.join(DATASET_PATH, "test/labels")

print("Train images:", len(os.listdir(TRAIN_IMAGES)))
print("Train labels:", len(os.listdir(TRAIN_LABELS)))
print("Val images:", len(os.listdir(VAL_IMAGES)))
print("Val labels:", len(os.listdir(VAL_LABELS)))
print("Test images:", len(os.listdir(TEST_IMAGES)))
print("Test labels:", len(os.listdir(TEST_LABELS)))

Train images: 20508
Train labels: 20508
Val images: 5821
Val labels: 5821
Test images: 2949
Test labels: 2949


## Explore class distribution

In [ ]:
import yaml
import os

data_yaml_path = os.path.join(DATASET_PATH, 'data.yaml')

with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

class_names = data['names']
print("Class Names:", class_names)
print("Number of classes:", len(class_names))

Class Names: ['adobo', 'almond-jelly', 'apple', 'apple-pie', 'babi-guling', 'bagel', 'bak-kut-teh', 'ball-shaped-bun-with-pork', 'barbecued-red-pork-in-sauce-with-rice', 'bean-curd-family-style', 'beef-bowl', 'beef-curry', 'beef-noodle-soup', 'bibimbap', 'boiled-chicken-and-vegetables', 'boiled-fish', 'boned-sliced-hainan-style-chicken-with-marinated-rice', 'braised-pork-meat-ball-with-napa-cabbage', 'broiled-eel-bowl', 'brownie', 'bubur-ayam', 'cabbage-roll', 'caesar-salad', 'champon', 'charcoal-boiled-pork-neck', 'chicken-cutlet', 'chicken-n-egg-on-rice', 'chicken-nugget', 'chicken-rice-curry-with-coconut', 'chilled-noodle', 'chop-suey', 'churro', 'clear-soup', 'coconut-milk-flavored-crepes-with-shrimp-and-beef', 'coconut-milk-soup', 'cold-tofu', 'crape', 'cream-puff', 'crispy-noodles', 'croissant', 'croquette', 'crullers', 'curry-puff', 'custard-tart', 'cutlet-curry', 'dipping-noodles', 'dish-consisting-of-stir-fried-potato-eggplant-and-green-pepper', 'doughnut', 'dried-fish', 'dry-

In [ ]:
from collections import defaultdict

class_counts = defaultdict(int)

for label_file in os.listdir(TRAIN_LABELS):
    with open(os.path.join(TRAIN_LABELS, label_file), 'r') as f:
        for line in f:
            parts = line.split()
            if parts: # Ensure the line is not empty
                class_id = int(parts[0])
                # Make sure class_id is within the bounds of the updated class_names
                if class_id < len(class_names):
                    class_name = class_names[class_id]
                    class_counts[class_name] += 1

print("Updated class counts in training data:")
for class_name, count in class_counts.items():
    print(f"  {class_name}: {count}")

Updated class counts in training data:
  miso-soup: 307
  hot-dog: 73
  croissant: 80
  apple-pie: 69
  hue-beef-rice-vermicelli-soup: 78
  rare-cheese-cake: 78
  khao-soi: 82
  clear-soup: 496
  mixed-rice: 455
  lamb-kebabs: 80
  pork-sticky-noodles: 81
  crullers: 77
  fried-noodle: 239
  omelet: 303
  parfait: 70
  rice: 238
  sandwiches: 118
  beef-curry: 164
  jjigae: 74
  soba-noodle: 102
  chilled-noodle: 78
  hot-and-sour-soup: 81
  nanbanzuke: 71
  curry-puff: 79
  tempura-udon: 90
  japanese-style-pancake: 93
  minestrone: 72
  coconut-milk-flavored-crepes-with-shrimp-and-beef: 76
  tempura-bowl: 95
  broiled-eel-bowl: 79
  stir-fried-beef: 216
  roll-bread: 87
  sashimi: 176
  mie-ayam: 72
  salmon: 212
  scone: 72
  eggplant-with-garlic-sauce: 79
  barbecued-red-pork-in-sauce-with-rice: 74
  bagel: 72
  fried-fish: 70
  thai-papaya-salad: 83
  ramen-noodle: 217
  takoyaki: 95
  fried-rice: 265
  twice-cooked-pork: 79
  stir-fried-chicken: 230
  boiled-fish: 71
  roast-duck

In [ ]:
low_image_classes = {}
for class_name, count in class_counts.items():
    if count < 75:
        low_image_classes[class_name] = count

print(f"Updated classes with fewer than 75 images ({len(low_image_classes)} classes):")
for class_name, count in low_image_classes.items():
    print(f"  {class_name}: {count} images")

total_images_in_low_image_classes = sum(low_image_classes.values())
print(f"Total images in these classes: {total_images_in_low_image_classes}")

Updated classes with fewer than 75 images (96 classes):
  hot-dog: 73 images
  apple-pie: 69 images
  parfait: 70 images
  jjigae: 74 images
  nanbanzuke: 71 images
  minestrone: 72 images
  mie-ayam: 72 images
  scone: 72 images
  barbecued-red-pork-in-sauce-with-rice: 74 images
  bagel: 72 images
  fried-fish: 70 images
  boiled-fish: 71 images
  stewed-pork-leg: 72 images
  tempura: 74 images
  cabbage-roll: 73 images
  bean-curd-family-style: 73 images
  scrambled-egg: 69 images
  tacos: 74 images
  caesar-salad: 72 images
  goya-chanpuru: 68 images
  steamed-egg-hotchpotch: 71 images
  kushikatu: 71 images
  simmered-pork: 73 images
  raisin-bread: 74 images
  trunip-pudding: 72 images
  potage: 73 images
  chicken-nugget: 71 images
  fried-spring-rolls: 72 images
  pizza-toast: 68 images
  crispy-noodles: 69 images
  french-toast: 71 images
  ginger-pork-saute: 70 images
  tortilla: 71 images
  champon: 74 images
  chop-suey: 70 images
  steamed-rice-roll: 70 images
  noodles-wit

## Drop meatloaf class

In [ ]:
meat_loaf_class_id = class_names.index('meat-loaf')
print(f"'meat-loaf' class ID: {meat_loaf_class_id}")

'meat-loaf' class ID: 108


In [ ]:
import shutil

image_files_to_delete = set()
label_files_to_delete = set()

# Iterate through all label files in the training data
for label_file_name in os.listdir(TRAIN_LABELS):
    label_file_path = os.path.join(TRAIN_LABELS, label_file_name)
    updated_lines = []
    contains_meat_loaf = False

    with open(label_file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        class_id = int(line.split()[0])
        if class_id == meat_loaf_class_id:
            contains_meat_loaf = True
        else:
            updated_lines.append(line)

    if contains_meat_loaf:
        if updated_lines:  # If other annotations remain, rewrite the file
            with open(label_file_path, 'w') as f:
                f.writelines(updated_lines)
        else:  # If no other annotations, mark label and image for deletion
            label_files_to_delete.add(label_file_path)
            image_file_name = label_file_name.replace('.txt', '.jpg')  # Assuming jpg images
            image_file_path = os.path.join(TRAIN_IMAGES, image_file_name)
            image_files_to_delete.add(image_file_path)

# Perform deletions for empty label files and corresponding images
for label_path in label_files_to_delete:
    if os.path.exists(label_path):
        os.remove(label_path)
        print(f"Deleted empty label file: {label_path}")

for image_path in image_files_to_delete:
    if os.path.exists(image_path):
        os.remove(image_path)
        print(f"Deleted corresponding image file: {image_path}")

Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_037550_jpg.rf.d492b5eab7fb6acdff49ab8e31064023.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_143343_jpg.rf.863abcde4c096305c5c700edaaccc725.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_090371_jpg.rf.5477c4a6768477d587b69508639a09be.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_093266_jpg.rf.e496b08f4a4e8251bdd7752cc557715b.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_093220_jpg.rf.0b1992160f5d677fee3802b2078959c6.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_091236_jpg.rf.d92047870b19b4d8aa5d1b1126cdf1a9.txt
Deleted empty label file: /content/V2-CareTech-Combined-Dataset-1/train/labels/meat_loaf_143242_jpg.rf.efed1346583fc69d14a9ae3356630188.txt
Deleted empty label 

In [ ]:
new_class_names = [name for name in class_names if name != 'meat-loaf']

# Update class IDs in label files if meat-loaf_class_id is not the last one
if meat_loaf_class_id < len(class_names) - 1:
    for label_file_name in os.listdir(TRAIN_LABELS):
        label_file_path = os.path.join(TRAIN_LABELS, label_file_name)
        updated_lines = []
        with open(label_file_path, 'r') as f:
            for line in f:
                parts = line.split()
                current_class_id = int(parts[0])
                if current_class_id > meat_loaf_class_id:
                    parts[0] = str(current_class_id - 1)
                updated_lines.append(' '.join(parts) + '\n')
        with open(label_file_path, 'w') as f:
            f.writelines(updated_lines)

# Update data.yaml
with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['names'] = new_class_names
data['nc'] = len(new_class_names)

with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"Updated data.yaml: nc is now {data['nc']} and 'meat-loaf' removed from names.")

# Update the global class_names variable
class_names = new_class_names

Updated data.yaml: nc is now 215 and 'meat-loaf' removed from names.


In [ ]:
from collections import defaultdict

class_counts = defaultdict(int)

for label_file in os.listdir(TRAIN_LABELS):
    with open(os.path.join(TRAIN_LABELS, label_file), 'r') as f:
        for line in f:
            parts = line.split()
            if parts: # Ensure the line is not empty
                class_id = int(parts[0])
                # Make sure class_id is within the bounds of the updated class_names
                if class_id < len(class_names):
                    class_name = class_names[class_id]
                    class_counts[class_name] += 1

print("Updated class counts in training data:")
for class_name, count in class_counts.items():
    print(f"  {class_name}: {count}")

Updated class counts in training data:
  miso-soup: 307
  hot-dog: 73
  croissant: 80
  apple-pie: 69
  hue-beef-rice-vermicelli-soup: 78
  rare-cheese-cake: 78
  khao-soi: 82
  clear-soup: 496
  mixed-rice: 455
  lamb-kebabs: 80
  pork-sticky-noodles: 81
  crullers: 77
  fried-noodle: 239
  omelet: 303
  parfait: 70
  rice: 238
  sandwiches: 118
  beef-curry: 164
  jjigae: 74
  soba-noodle: 102
  chilled-noodle: 78
  hot-and-sour-soup: 81
  nanbanzuke: 71
  curry-puff: 79
  tempura-udon: 90
  japanese-style-pancake: 93
  minestrone: 72
  coconut-milk-flavored-crepes-with-shrimp-and-beef: 76
  tempura-bowl: 95
  broiled-eel-bowl: 79
  stir-fried-beef: 216
  roll-bread: 87
  sashimi: 176
  mie-ayam: 72
  salmon: 212
  scone: 72
  eggplant-with-garlic-sauce: 79
  barbecued-red-pork-in-sauce-with-rice: 74
  bagel: 72
  fried-fish: 70
  thai-papaya-salad: 83
  ramen-noodle: 217
  takoyaki: 95
  fried-rice: 265
  twice-cooked-pork: 79
  stir-fried-chicken: 230
  boiled-fish: 71
  roast-duck

## Data augmentation: random erasing
In each of the 96 classes that have fewer than 75 images, I randomly selected 50% of the images from each class and generate an edited version of that image that has a random rectangle in the image that is masked. This improves model robustness. (The original image is not deleted.)

In [ ]:
import cv2
import numpy as np
import os
import random
import shutil

def random_erase(image, sl=0.02, sh=0.4, r1=0.3, r2=1/0.3, fill_mode='random'):
    # Convert image to NumPy array for easier manipulation
    img_np = np.array(image)
    img_h, img_w, _ = img_np.shape
    img_area = img_h * img_w

    while True:
        se = random.uniform(sl, sh) * img_area
        re = random.uniform(r1, r2)

        er_h = int(np.sqrt(se * re))
        er_w = int(np.sqrt(se / re))

        if er_w < img_w and er_h < img_h:
            x = random.randint(0, img_w - er_w)
            y = random.randint(0, img_h - er_h)

            if fill_mode == 'random':
                # Fill with random pixel values
                img_np[y:y + er_h, x:x + er_w, :] = np.random.randint(0, 256, size=(er_h, er_w, 3))
            elif fill_mode == 'mean':
                # Fill with mean pixel value of the erased region or entire image
                # For simplicity, filling with mean of the erased region (can be adapted for global mean)
                mean_val = np.mean(img_np[y:y + er_h, x:x + er_w, :], axis=(0, 1))
                img_np[y:y + er_h, x:x + er_w, :] = mean_val.astype(np.uint8)
            else: # 'black' or any other value
                img_np[y:y + er_h, x:x + er_w, :] = 0  # Fill with black

            return img_np

    return img_np # Should not reach here if loop is infinite until valid erase

print("Random erase function defined.")

Random erase function defined.


In [ ]:
augmented_images_count = 0

for class_name, count in low_image_classes.items():
    if count < 75: # Double-check the condition for safety
        # Get all images for the current class in the training set
        class_image_files = []
        for img_file in os.listdir(TRAIN_IMAGES):
            label_file = img_file.replace('.jpg', '.txt') # Assuming .jpg images and .txt labels
            label_path = os.path.join(TRAIN_LABELS, label_file)

            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.split()
                        if parts:
                            class_id_in_label = int(parts[0])
                            current_class_index = class_names.index(class_name)
                            if class_id_in_label == current_class_index:
                                class_image_files.append(img_file)
                                break # Found the class, move to next image file

        # Select 50% of these images for augmentation
        num_images_to_augment = int(len(class_image_files) * 0.5)
        images_to_augment = random.sample(class_image_files, num_images_to_augment)

        for original_img_filename in images_to_augment:
            original_img_path = os.path.join(TRAIN_IMAGES, original_img_filename)
            original_label_filename = original_img_filename.replace('.jpg', '.txt')
            original_label_path = os.path.join(TRAIN_LABELS, original_label_filename)

            # Load original image
            img = cv2.imread(original_img_path)
            if img is None:
                print(f"Warning: Could not load image {original_img_path}. Skipping augmentation for this image.")
                continue

            # Apply random erasing
            augmented_img = random_erase(img.copy()) # Use a copy to avoid modifying original in memory

            # Generate new filename for augmented image and label
            base_name, ext = os.path.splitext(original_img_filename)
            augmented_img_filename = f"{base_name}_aug{ext}"
            augmented_label_filename = f"{base_name}_aug.txt"

            augmented_img_path = os.path.join(TRAIN_IMAGES, augmented_img_filename)
            augmented_label_path = os.path.join(TRAIN_LABELS, augmented_label_filename)

            # Save augmented image
            cv2.imwrite(augmented_img_path, augmented_img)

            # Copy original label to new augmented label file
            shutil.copy(original_label_path, augmented_label_path)
            augmented_images_count += 1

print(f"Finished random erasing. Total augmented images created: {augmented_images_count}")

Finished random erasing. Total augmented images created: 3374


### Check that images were added to dataset

In [ ]:
print("Train images:", len(os.listdir(TRAIN_IMAGES)))
print("Train labels:", len(os.listdir(TRAIN_LABELS)))
print("Val images:", len(os.listdir(VAL_IMAGES)))
print("Val labels:", len(os.listdir(VAL_LABELS)))
print("Test images:", len(os.listdir(TEST_IMAGES)))
print("Test labels:", len(os.listdir(TEST_LABELS)))

Train images: 23803
Train labels: 23803
Val images: 5821
Val labels: 5821
Test images: 2949
Test labels: 2949


In [ ]:
!zip -r V2-CareTech-Combined-Dataset-1-Random-Erasing.zip /content/V2-CareTech-Combined-Dataset-1

Streaming output truncated to the last 5000 lines.
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/hue_beef_rice_vermicelli_soup_167294_jpg.rf.da268e4d1ff5d4d745cd5bbb108b5e90.txt (deflated 40%)
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/stew_016753_jpg.rf.00b0acfa7a992f34e2f6b46f22304075.txt (deflated 51%)
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/hambarg_steak_006029_jpg.rf.1aaddc2548524fc090688bc24561b74a.txt (deflated 13%)
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/nasi_padang_245602_jpg.rf.83b57c658ef2aeea55ae79418171dd80.txt (deflated 29%)
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/dish_consisting_of_stir-fried_potato_eggplant_and_green_pepper_046004_jpg.rf.32492991d96d8d024082e87ed091f71f.txt (deflated 29%)
  adding: content/V2-CareTech-Combined-Dataset-1/train/labels/hamburger_001594_jpg.rf.2751f123eb3c10243d8c7970c445001b.txt (deflated 17%)
  adding: content/V2-CareTech-Combined-Dataset-